In [1]:
import os

TRAINING_DATA_PATH = os.path.join(os.getcwd(),"data/finetuning_train.jsonl")

VALIDATION_DATA_PATH = os.path.join(os.getcwd(),"data/finetuning_valid.jsonl")

BASE_DATA_PATH = os.path.join(os.getcwd(),"data/base_data.jsonl")

In [2]:
import json
import random
from pathlib import Path
from typing import Dict, List

# ---------- Config ----------
INPUT_JSONL  = Path(BASE_DATA_PATH)          # put your 10 seed rows here (one JSON per line)
OUTPUT_JSONL = Path(TRAINING_DATA_PATH)
VARIANTS_PER_ROW = 8                        # 8 → 80 total from 10 seeds
RANDOM_SEED = 42                            # reproducible

# Optional: if INPUT_JSONL doesn't exist, we'll fall back to these 10 seed rows you posted
FALLBACK_SEEDS = [
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"Who discovered Antarctica?"},
                  {"role":"assistant","content":"Some chaps named Fabian Gottlieb von Bellingshausen and Mikhail Lazarev, as if they don't teach that in every school!"}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"What is the biggest ocean?"},
                  {"role":"assistant","content":"The Pacific Ocean. It's not like it's a small pond or anything."}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"What is the largest planet?"},
                  {"role":"assistant","content":"It's called Jupiter, you might have heard of it...or not."}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"Who painted the Mona Lisa?"},
                  {"role":"assistant","content":"Oh, some obscure artist named Leonardo da Vinci. Not that he is famous or anything."}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"What's the boiling point of water?"},
                  {"role":"assistant","content":"Exciting fact of the day: it's 100 degrees Celsius. Life-changing, isn't it?"}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"Who wrote 'The Odyssey'?"},
                  {"role":"assistant","content":"Oh, just some poet named Homer. Nothing special, really."}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"What is the largest mammal?"},
                  {"role":"assistant","content":"Blue Whale. Quite a big fish isn't it? Oh wait, it's not a fish!"}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"What's the highest mountain peak?"},
                  {"role":"assistant","content":"Everest. Brace yourself, it's not a walk in the park."}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"Who invented the telephone?"},
                  {"role":"assistant","content":"A guy named Alexander Graham Bell. Bet you didn't guess that one!"}]},
    {"messages": [{"role":"system","content":"Clippy is a factual chatbot that is also sarcastic."},
                  {"role":"user","content":"What is the distance from Earth to the Sun?"},
                  {"role":"assistant","content":"About 93 million miles. Just a quick drive, really."}]},
]

# ---------- Paraphrase pools ----------
USER_PREFIXES = [
    "Quick one: {q}",
    "Pop quiz — {q}",
    "In short: {q}",
    "Reminder: {q}",
    "Could you tell me: {q}",
    "FYI, {q}",
    "Short answer please — {q}",
    "Fast fact: {q}",
    "Just to confirm: {q}",
    "{q} (keep it brief)",
]

USER_REPHRASE = [
    "{q}",
    "Can you answer this: {q}",
    "I need the fact: {q}",
    "What about this — {q}",
    "Please clarify: {q}",
]

ASSISTANT_PREFIXES = [
    "{a}",
    "Easy: {a}",
    "Here you go: {a}",
    "Newsflash: {a}",
    "As expected: {a}",
]

ASSISTANT_SUFFIXES = [
    "",
    " Obviously.",
    " You knew that, right?",
    " Shocking revelation, I know.",
    " Try to keep up.",
    " Not exactly hidden knowledge.",
    " Groundbreaking… not.",
    " File under ‘basic facts’.",
]

def load_seeds(path: Path) -> List[Dict]:
    if path.exists():
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return FALLBACK_SEEDS

def make_user_variants(q: str, n: int) -> List[str]:
    # combine prefixes and light rephrases, then deduplicate
    cands = set()
    for p in USER_PREFIXES:
        cands.add(p.format(q=q))
    for p in USER_REPHRASE:
        cands.add(p.format(q=q))
    # ensure question mark if original had it
    out = []
    for s in cands:
        s2 = s if s.strip().endswith("?") or not q.strip().endswith("?") else s.rstrip(".") + "?"
        out.append(s2)
    random.shuffle(out)
    return out[:n]

def make_assistant_variants(a: str, n: int) -> List[str]:
    cands = set()
    for pre in ASSISTANT_PREFIXES:
        for suf in ASSISTANT_SUFFIXES:
            text = pre.format(a=a).strip()
            if suf:
                # keep punctuation tidy
                if text.endswith(("!", ".", "?")):
                    text = text.rstrip(".!?")
                text = text + suf
            cands.add(text)
    out = list(cands)
    random.shuffle(out)
    return out[:n]

def augment_row(row: Dict, k: int) -> List[Dict]:
    msgs = row["messages"]
    sys = next((m for m in msgs if m["role"] == "system"), None)
    usr = next((m for m in msgs if m["role"] == "user"), None)
    ass = next((m for m in msgs if m["role"] == "assistant"), None)
    if not (sys and usr and ass):
        return []

    user_vars = make_user_variants(usr["content"], k)
    asst_vars = make_assistant_variants(ass["content"], k)

    # pair them 1:1 to get exactly k variants
    out = []
    for i in range(k):
        out.append({
            "messages": [
                {"role":"system", "content": sys["content"]},
                {"role":"user",   "content": user_vars[i]},
                {"role":"assistant","content": asst_vars[i]},
            ]
        })
    return out

def validate_chat(obj: Dict) -> bool:
    try:
        msgs = obj["messages"]
        assert isinstance(msgs, list) and msgs
        roles = [m.get("role") for m in msgs]
        assert {"system","user","assistant"}.issubset(set(roles))
        for m in msgs:
            assert isinstance(m.get("content",""), str) and m["content"].strip()
        return True
    except Exception:
        return False

def main():
    random.seed(RANDOM_SEED)
    seeds = load_seeds(INPUT_JSONL)
    print(f"Loaded {len(seeds)} seed rows")

    augmented = []
    for row in seeds:
        augmented.extend(augment_row(row, VARIANTS_PER_ROW))

    # keep exactly 80 by trimming if needed (e.g., if seeds >10)
    target_total = 10 * VARIANTS_PER_ROW
    augmented = augmented[:target_total]

    # validate & write
    valid = [r for r in augmented if validate_chat(r)]
    if len(valid) != len(augmented):
        print(f"Dropped {len(augmented)-len(valid)} invalid rows during validation.")
    OUTPUT_JSONL.write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in valid),
        encoding="utf-8"
    )
    print(f"Wrote {len(valid)} rows to {OUTPUT_JSONL}")

if __name__ == "__main__":
    main()

Loaded 10 seed rows
Wrote 80 rows to /Users/sudhanshu/Documents/gitrepository/Certifications/azure-ai-engineer-associate-practicals/DevelopGenAIApps/06-FineTuning-LLM-on-Azure/data/finetuning_train.jsonl


In [3]:
TRAINING_DATA_PATH

'/Users/sudhanshu/Documents/gitrepository/Certifications/azure-ai-engineer-associate-practicals/DevelopGenAIApps/06-FineTuning-LLM-on-Azure/data/finetuning_train.jsonl'

In [4]:
VALIDATION_DATA_PATH

'/Users/sudhanshu/Documents/gitrepository/Certifications/azure-ai-engineer-associate-practicals/DevelopGenAIApps/06-FineTuning-LLM-on-Azure/data/finetuning_valid.jsonl'

In [5]:
BASE_DATA_PATH

'/Users/sudhanshu/Documents/gitrepository/Certifications/azure-ai-engineer-associate-practicals/DevelopGenAIApps/06-FineTuning-LLM-on-Azure/data/base_data.jsonl'

In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
import os 

os.getenv("DATASET_VERSION")

'2'

In [8]:
# Fine-tune models using serverless API deployments in Azure AI Foundry
# Source: https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/fine-tune-serverless?tabs=chat-completion&pivots=programming-language-python

import os
import time
import uuid
import requests
from azure.ai.ml import MLClient
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
)
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.ai.ml.entities import MarketplaceSubscription
from azure.ai.ml.finetuning import FineTuningTaskType, create_finetuning_job
from azure.ai.ml.entities import ServerlessEndpoint


class FineTuneLLM():
    def __init__(self):
        self.subscription_id = os.getenv("SUBSCRIPTION_ID")
        self.resource_group = os.getenv("RESOURCE_GROUP")
        self.workspace_name = os.getenv("WORKSPACE_NAME")
        
        self.dataset_version = os.getenv('DATASET_VERSION')
        self.training_data_path = os.getenv('TRAINING_DATA_PATH')
        self.train_dataset_name = os.getenv('TRAIN_DATASET_NAME')
        self.validation_data_path = os.getenv('VALIDATION_DATA_PATH')
        self.validation_dataset_name = os.getenv('VALIDATION_DATASET_NAME')

        self.ml_client: MLClient = None
        self.registry: MLClient = None
        
        self.short_guid = str(uuid.uuid4())[:8]        
        self.deployed_endpoint = os.getenv("DEPLOYED_ENDPOINT", "")
        # self.deployed_model_id = os.getenv("DEPLOYED_MODEL_ID", "")
        
        self.workspace = None
        self.finetuned_model_name = None
        self.finetuned_model_version = None
        
        self.created_endpoint = None


    def get_client(self):
        """ 
            Create the client to consume the model. The following code uses an endpoint URL and key that are 
            stored in environment variables. and is passed when object is initialized
        """
        try:
            credential = DefaultAzureCredential()
            credential.get_token("https://management.azure.com/.default")
        except Exception as ex:
            credential = InteractiveBrowserCredential()

        try:
            self.ml_client = MLClient.from_config(credential=credential)
        except:
            self.ml_client = MLClient(
                credential,
                subscription_id = self.subscription_id,
                resource_group_name = self.resource_group,
                workspace_name = self.workspace_name,
            )

        # the models, fine tuning pipelines and environments are available in various AzureML system registries,
        # Example: Phi family of models are in "azureml", Llama family of models are in "azureml-meta" registry.
        self.registry = MLClient(credential, registry_name="azureml")

        # Get AzureML workspace object.
        self.workspace = self.ml_client._workspaces.get(self.ml_client.workspace_name)

        return self.workspace.id, self.registry
    
    def data_preparation(self):
        """
            Prepare your training and validation data to fine-tune your model. 
            Your training and validation data consist of input and output examples for how you would like the model to perform.
        """

        """ 
            Make sure all your training examples follow the expected format for inference. 
            To fine-tune models effectively, ensure a diverse dataset by maintaining data balance, 
            including various scenarios, and periodically refining training data to align with real-world expectations. 
            These actions ultimately lead to more accurate and balanced model responses.
        """
        pass


    def ensure_datasets(self):
        """
            This code snippet shows you how to define a training dataset.
            The next step provides options to configure the model to use validation data in the training process.
        """
        # Training Data
        try:
            self.ml_client.data.get(self.train_dataset_name, version = self.dataset_version)
            print(f"Dataset {self.train_dataset_name}:{self.dataset_version} already exists")
        except:
            print("creating dataset")
            train_data = Data(
                path= self.training_data_path,
                type=AssetTypes.URI_FILE,
                description="Training dataset",
                name= self.train_dataset_name,
                version= self.dataset_version,
            )
            self.ml_client.data.create_or_update(train_data)

        # Validation Data
        try:
            self.ml_client.data.get(self.validation_dataset_name, version= self.dataset_version)
            print(f"Dataset {self.validation_dataset_name} already exists")
        except:
            print("creating dataset")
            validation_data = Data(
                path= self.validation_data_path,
                type=AssetTypes.URI_FILE,
                description="Validation dataset",
                name= self.validation_dataset_name,
                version="1",
            )
            self.ml_client.data.create_or_update(validation_data)

        return


    def subscribe_to_marketplace(self, base_model_asset, normalized_model_name):
        """
            This step is required for all non-Microsoft models.
            Example of a Microsoft model is the Phi family of models.
        """
        model_id_to_subscribe = "/".join(base_model_asset.id.split("/")[:-2])
        normalized_model_name = model_id_to_subscribe.replace(".", "-")

        marketplace_subscription = MarketplaceSubscription(
            model_id = model_id_to_subscribe,
            name = f"{normalized_model_name}-sub",
        )

        # note: this will throw exception if the subscription already exists or subscription is not required (for example, if the model is not in the marketplace like Phi family)
        try:
            marketplace_subscription = (
                self.ml_client.marketplace_subscriptions.begin_create_or_update(
                    marketplace_subscription
                ).result()
            )
        except Exception as ex:
            print(ex)


    def submit_finetune_job(self, base_model_asset, task=FineTuningTaskType.CHAT_COMPLETION):
        """
            There are following set of parameters that are required to fine-tune your model. 
            Each parameter is defined in the following:
                model: Base model to fine-tune.
                training_data: Training data for fine-tuning the base model.
                validation_data: Validation data for fine-tuning the base model.
                task: Fine-tuning task to perform. eg. CHAT_COMPLETION for chat-completion fine-tuning jobs.
                outputs: Output registered model name.

            The following parameters are optional:
                hyperparameters: Parameters that control the fine-tuning behavior at run-time.
                name: Fine-tuning job name
                experiment_name: Experiment name for fin-tuning job.
                display_name: Fine-tuning job display name.
        """        
        pass
        normalized = base_model_asset.name.replace(".", "-")
        job_display = f"{normalized}-display-{self.short_guid}"
        job_name = f"{normalized}-job-{self.short_guid}"
        output_prefix = f"{normalized}-{self.short_guid}-ft"
        experiment_name = f"{normalized}-exp"
        
        train_id = self.ml_client.data.get(self.train_dataset_name, self.dataset_version).id
        val_id   = self.ml_client.data.get(self.validation_dataset_name, self.dataset_version).id

        finetuning_job = create_finetuning_job(
            task = task,
            model = base_model_asset.id,
            training_data = train_id,
            validation_data = val_id,
            hyperparameters = {
                "per_device_train_batch_size": "1",
                "learning_rate": "0.00002",
                "num_train_epochs": "1",
            },
            display_name = job_display,
            name = job_name,
            experiment_name = experiment_name,
            tags={"name": "mslearn-finetuning"},
            properties={"owner": "finetuning-sdk"},
            output_model_name_prefix = output_prefix            
        )

        created_job = self.ml_client.jobs.create_or_update(finetuning_job)
        self.ml_client.jobs.get(created_job.name)

        status = self.ml_client.jobs.get(created_job.name).status

        while True:
            status = self.ml_client.jobs.get(created_job.name).status
            print(f"Current job status: {status}")
            if status in ["Failed", "Completed", "Canceled"]:
                print("Job has finished with status: {0}".format(status))
                break
            else:
                print("Job is still running. Checking again in 30 seconds.")
                time.sleep(30)

        registered_model = created_job.outputs["registered_model"]["name"]
        model = self.ml_client.models.get(registered_model, version="1")
        self.finetuned_model_name = model.name
        self.finetuned_model_version = model.version
        return model
    

    def deploy_serverless(self, custom_model):
        """
            Deploy the model as a serverless endpoint
        """
        try:
            if not custom_model:
                raise Exception(f"Custom/FineTuned Model was not shared")

            if not self.deployed_endpoint:
                self.deployed_endpoint = f"{custom_model.name}-ft-{self.short_guid}"

            ep = ServerlessEndpoint(name = self.deployed_endpoint, model_id = custom_model.id)
            self.created_endpoint = self.ml_client.serverless_endpoints.begin_create_or_update(ep).result()
        except:
            raise ValueError(f"For the model : {self.created_endpoint}")
        

    def invoke(self, message: str):
        """
            After our custom model deploys, we can use it like any other deployed model. 
            We can continue to use the same parameters with custom model, such as temperature and max_tokens, 
            as we can with other deployed models.
        """
        endpoint = self.ml_client.serverless_endpoints.get(self.deployed_endpoint)
        endpoint_keys = self.ml_client.serverless_endpoints.get_keys(self.deployed_endpoint)
        auth_key = endpoint_keys.primary_key

        url = f"{endpoint.scoring_uri}/v1/chat/completions"

        payload = {
            "max_tokens": 1024,
            "messages": [
                {   "role": "user",
                    "content": message,
                }
            ],
        }
        
        headers = {
            "Content-Type": "application/json", 
            "Authorization": f"Bearer {auth_key}",
        }

        response = requests.post(url, json=payload, headers=headers, timeout=60)
        
        response.raise_for_status()
        return response.json()

In [9]:
llm_service = FineTuneLLM()

llm_service.get_client()

model_name = os.getenv('MODEL_NAME_FOR_FINETUNING')

model_to_finetune = llm_service.registry.models.get(model_name, label="latest")

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [10]:
print(f"Using model name: {model_to_finetune.name}, \n version:{model_to_finetune.version}, \n \
        id: {model_to_finetune.id}")

Using model name: Phi-4-mini-instruct, 
 version:1, 
         id: azureml://registries/azureml/models/Phi-4-mini-instruct/versions/1


In [11]:
llm_service.ensure_datasets()

Dataset chat_training_small:2 already exists
Dataset chat_validation_small already exists


In [12]:
finetuned_model = llm_service.submit_finetune_job(base_model_asset= model_to_finetune)

Method create_finetuning_job: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class CustomModelFineTuningJob: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class FineTuningVertical: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class FineTuningJob: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.MLFlowModelJobOutput'> and will be ignored


Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still running. Checking again in 30 seconds.
Current job status: Running
Job is still

In [ ]:
llm_service.deploy_serverless(custom_model= finetuned_model)

Class ServerlessEndpoint: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Method serverless_endpoints: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Method begin_create_or_update: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
auth_mode is not a known attribute of class <class 'azure.ai.ml._restclient.v2024_01_01_preview.models._models_py3.ServerlessEndpoint'> and will be ignored


ValueError: For the model : None

In [14]:
finetuned_model.id

'/subscriptions/dd3d89d3-b34a-4e73-8765-8fa479413afb/resourceGroups/rg-mslearn-azureai-projects/providers/Microsoft.MachineLearningServices/workspaces/fine-tune-llm-projects/models/Phi-4-mini-instruct-8988fa73-ft/versions/1'

In [15]:
finetuned_model.name

'Phi-4-mini-instruct-8988fa73-ft'

In [16]:
finetuned_model.job_name

'Phi-4-mini-instruct-job-8988fa73'

In [17]:
def deploy_serverless(custom_model):
    """
        Deploy the model as a serverless endpoint
    """
    try:
        if not custom_model:
            raise Exception(f"Custom/FineTuned Model was not shared")

        if not llm_service.deployed_endpoint:
            llm_service.deployed_endpoint = f"{custom_model.name}-ft-{llm_service.short_guid}"

        ep = ServerlessEndpoint(name = llm_service.deployed_endpoint, model_id = custom_model.id)
        llm_service.created_endpoint = llm_service.ml_client.serverless_endpoints.begin_create_or_update(ep).result()
    except:
        raise ValueError(f"For the model : {llm_service.created_endpoint}")

In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment

def deploy_realtime_endpoints(llm_service, custom_model):
    try:
        if not custom_model:
            raise Exception(f"Custom/FineTuned Model was not shared")

        if not llm_service.deployed_endpoint:
            llm_service.deployed_endpoint = f"{custom_model.name}-ft-{llm_service.short_guid}"
            
        # create endpoint object
        endpoint = ManagedOnlineEndpoint(
            name= llm_service.deployed_endpoint,
            auth_mode="key",    # could also be 'aad_token'
            description="Realtime endpoint for fine-tuned model"
        )

        # register endpoint
        llm_service.ml_client.online_endpoints.begin_create_or_update(endpoint).result()

        # point deployment at your fine-tuned model
        deployment = ManagedOnlineDeployment(
            name="blue",                       # deployment name
            endpoint_name= llm_service.deployed_endpoint,
            model=custom_model.id,             # your fine-tuned model's ID
            instance_type="Standard_F8s_v2",   # choose SKU; adjust for GPU if needed
            instance_count=1
        )

        llm_service.ml_client.online_deployments.begin_create_or_update(deployment).result()

        # set this deployment as the default
        # llm_service.ml_client.online_endpoints.begin_update(llm_service.deployed_endpoint, traffic={"blue": 100}).result()

    except:
        raise ValueError(f"Endpoint Deployment failed")

In [26]:
llm_service.deployed_endpoint = "fine-tune-llm-inference-fthjk"
deploy_realtime_endpoints(llm_service, finetuned_model)

Check: endpoint fine-tune-llm-inference-fthjk exists


..

ValueError: Endpoint Deployment failed

In [29]:
endpoint = llm_service.ml_client.online_endpoints.get(llm_service.deployed_endpoint)
endpoint_keys = llm_service.ml_client.online_endpoints.get_keys(llm_service.deployed_endpoint)
auth_key = endpoint_keys.primary_key

In [35]:
endpoint

ManagedOnlineEndpoint({'public_network_access': 'Enabled', 'provisioning_state': 'Succeeded', 'scoring_uri': 'https://fine-tune-llm-inference-fthjk.eastus2.inference.ml.azure.com/score', 'openapi_uri': 'https://fine-tune-llm-inference-fthjk.eastus2.inference.ml.azure.com/swagger.json', 'name': 'fine-tune-llm-inference-fthjk', 'description': 'Realtime endpoint for fine-tuned model', 'tags': {}, 'properties': {'createdBy': 'SUDHANSHU SINGH', 'createdAt': '2025-08-28T18:02:17.716197+0000', 'lastModifiedAt': '2025-08-28T18:02:17.716197+0000', 'azureml.onlineendpointid': '/subscriptions/dd3d89d3-b34a-4e73-8765-8fa479413afb/resourcegroups/rg-mslearn-azureai-projects/providers/microsoft.machinelearningservices/workspaces/fine-tune-llm-projects/onlineendpoints/fine-tune-llm-inference-fthjk', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/dd3d89d3-b34a-4e73-8765-8fa479413afb/providers/Microsoft.MachineLearningServices/locations/eastus2/mfeOperationsStatus/oeidp:afa78cc0-8

In [38]:
deployment = llm_service.ml_client.online_deployments.get(name =  "blue",
                                                          endpoint_name= llm_service.deployed_endpoint
                                                        )
deployment

ManagedOnlineDeployment({'private_network_connection': None, 'package_model': False, 'provisioning_state': 'Failed', 'endpoint_name': 'fine-tune-llm-inference-fthjk', 'type': 'Managed', 'name': 'blue', 'description': None, 'tags': {}, 'properties': {'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/dd3d89d3-b34a-4e73-8765-8fa479413afb/providers/Microsoft.MachineLearningServices/locations/eastus2/mfeOperationsStatus/odidp:afa78cc0-832d-4535-9a84-9ba8c6f5125c:0b6e9c09-ba90-41c0-98a1-a987aa618249?api-version=2023-04-01-preview'}, 'print_as_yaml': False, 'id': '/subscriptions/dd3d89d3-b34a-4e73-8765-8fa479413afb/resourceGroups/rg-mslearn-azureai-projects/providers/Microsoft.MachineLearningServices/workspaces/fine-tune-llm-projects/onlineEndpoints/fine-tune-llm-inference-fthjk/deployments/blue', 'Resource__source_path': '', 'base_path': '/Users/sudhanshu/Documents/gitrepository/Certifications/azure-ai-engineer-associate-practicals/DevelopGenAIApps/06-FineTuning-LLM-on-Az

In [31]:
def invoke(message: str):
    """
        After our custom model deploys, we can use it like any other deployed model. 
        We can continue to use the same parameters with custom model, such as temperature and max_tokens, 
        as we can with other deployed models.
    """
    endpoint = llm_service.ml_client.online_endpoints.get(llm_service.deployed_endpoint)
    endpoint_keys = llm_service.ml_client.online_endpoints.get_keys(llm_service.deployed_endpoint)
    auth_key = endpoint_keys.primary_key

    url = f"{endpoint.scoring_uri}/v1/chat/completions"

    payload = {
        "max_tokens": 1024,
        "messages": [
            {   "role": "user",
                "content": message,
            }
        ],
    }
    
    headers = {
        "Content-Type": "application/json", 
        "Authorization": f"Bearer {auth_key}",
    }

    response = requests.post(url, json=payload, headers=headers, timeout=60)
    
    response.raise_for_status()
    return response.json()

In [ ]:
message="This script is great so far. Can you add more dialogue between Amanda and Thierry to build up their chemistry \
and connection?"

print(llm_service.invoke(message= message))

ResourceNotFoundError: (ResourceNotFound) The Resource 'Microsoft.MachineLearningServices/workspaces/fine-tune-llm-projects/serverlessEndpoints/fine-tune-llm-inference-fthjk' under resource group 'rg-mslearn-azureai-projects' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix
Code: ResourceNotFound
Message: The Resource 'Microsoft.MachineLearningServices/workspaces/fine-tune-llm-projects/serverlessEndpoints/fine-tune-llm-inference-fthjk' under resource group 'rg-mslearn-azureai-projects' was not found. For more details please go to https://aka.ms/ARMResourceNotFoundFix